In [1]:
import geopandas as gpd
import folium
from folium import plugins
import pandas as pd

# Load the London Plan consultation shapefile
shp_path = r"C:\Users\kylec\Downloads\lp-consultation-oct-2009-inner-outer-london-shp\lp-consultation-oct-2009-inner-outer-london.shp"

london_boundaries = gpd.read_file(shp_path)

print(f"Loaded {len(london_boundaries)} boundaries")
print(f"CRS: {london_boundaries.crs}")
print(f"\nColumns: {london_boundaries.columns.tolist()}")
print(f"\nBoundaries:")
print(london_boundaries[['Boundary', 'Source', 'Area_Ha']])

Loaded 2 boundaries
CRS: PROJCS["OSGB36 / British National Grid",GEOGCS["OSGB36",DATUM["Ordnance_Survey_of_Great_Britain_1936",SPHEROID["Airy 1830",6377563.396,299.3249646,AUTHORITY["EPSG","7001"]],AUTHORITY["EPSG","6277"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-2],PARAMETER["scale_factor",0.999601272],PARAMETER["false_easting",400000],PARAMETER["false_northing",-100000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Columns: ['Boundary', 'Source', 'Area_Ha', 'Shape_Leng', 'Shape_Area', 'geometry']

Boundaries:
       Boundary                          Source        Area_Ha
0  Inner London  London Plan Consultation Draft   34863.295694
1  Outer London  London Plan Consultation Draft  124606.812118


In [2]:
# Reproject CRS to WGS84 for folium
print(f"Current CRS: {london_boundaries.crs}")

# Reproject to WGS84 (EPSG:4326) for folium compatibility
if london_boundaries.crs != 'EPSG:4326':
    london_boundaries = london_boundaries.to_crs('EPSG:4326')
    print(f"Reprojected to: {london_boundaries.crs}")
else:
    print("Already in WGS84")

Current CRS: PROJCS["OSGB36 / British National Grid",GEOGCS["OSGB36",DATUM["Ordnance_Survey_of_Great_Britain_1936",SPHEROID["Airy 1830",6377563.396,299.3249646,AUTHORITY["EPSG","7001"]],AUTHORITY["EPSG","6277"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-2],PARAMETER["scale_factor",0.999601272],PARAMETER["false_easting",400000],PARAMETER["false_northing",-100000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Reprojected to: EPSG:4326


In [3]:
# Calculate the center of London for the map
bounds = london_boundaries.total_bounds
center_lon = (bounds[0] + bounds[2]) / 2
center_lat = (bounds[1] + bounds[3]) / 2

print(f"Map center: ({center_lat}, {center_lon})")
print(f"Bounds: {bounds}")

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='OpenStreetMap'
)

print("Map created, adding boundaries...")

Map center: (51.4893171330082, -0.08817978285719)
Bounds: [-0.51037507 51.28676013  0.33401551 51.69187413]
Map created, adding boundaries...


In [ ]:
# Load borough boundaries from MSOA shapefiles
import glob

print("Loading MSOA shapefiles to create borough boundaries...")
shapefile_dir = r"C:\Users\kylec\Downloads\LB_MSOA2021_shp\msoa2021"
shapefiles = sorted(glob.glob(f"{shapefile_dir}/*.shp"))

gdfs = []
for shp in shapefiles:
    gdf = gpd.read_file(shp)
    gdfs.append(gdf)

# Combine all MSOAs
all_msoas = pd.concat(gdfs, ignore_index=True)

# Reproject to WGS84 if needed
if all_msoas.crs != 'EPSG:4326':
    all_msoas = all_msoas.to_crs('EPSG:4326')

# Dissolve by borough (lad22nm) to create borough boundaries
borough_boundaries = all_msoas.dissolve(by='lad22nm', as_index=False)
print(f"Created {len(borough_boundaries)} borough boundaries")

# Add borough boundaries to map (dashed gray lines)
for idx, row in borough_boundaries.iterrows():
    folium.GeoJson(
        data=gpd.GeoSeries([row.geometry]).__geo_interface__,
        style_function=lambda x: {
            'fillColor': 'none',
            'color': '#999999',
            'weight': 1,
            'fillOpacity': 0,
            'dashArray': '3, 3'
        },
        popup=folium.Popup(f"<b>Borough</b>", max_width=200)
    ).add_to(m)

print("Borough boundaries added")

# Define colors for Inner and Outer London
def get_color(boundary_type):
    if boundary_type == 'Inner London':
        return '#FF6B6B'  # Red
    else:
        return '#4D96FF'  # Blue

# Add Inner/Outer London boundaries
for idx, row in london_boundaries.iterrows():
    boundary_type = row['Boundary']
    color = get_color(boundary_type)
    
    folium.GeoJson(
        data=gpd.GeoSeries([row.geometry]).__geo_interface__,
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': c,
            'weight': 2,
            'fillOpacity': 0.2
        },
        highlight_function=lambda x, c=color: {
            'fillColor': c,
            'color': c,
            'weight': 3,
            'fillOpacity': 0.5
        },
        popup=folium.Popup(
            f"<b>{boundary_type}</b><br>"
            f"Area: {row['Area_Ha']:,.0f} Ha<br>"
            f"Source: {row['Source']}",
            max_width=300
        ),
        name=boundary_type
    ).add_to(m)

print("Inner/Outer London boundaries added")

In [ ]:
# Load and dissolve MSOA shapefiles to get borough boundaries
import glob
from pathlib import Path

print("Loading MSOA shapefiles to create borough boundaries...")
shapefile_dir = r"C:\Users\kylec\Downloads\LB_MSOA2021_shp\msoa2021"
shapefiles = sorted(glob.glob(f"{shapefile_dir}/*.shp"))

gdfs = []
for shp in shapefiles:
    gdf = gpd.read_file(shp)
    gdfs.append(gdf)

# Combine all MSOAs
all_msoas = pd.concat(gdfs, ignore_index=True)

# Reproject to WGS84 if needed
if all_msoas.crs != 'EPSG:4326':
    all_msoas = all_msoas.to_crs('EPSG:4326')

# Dissolve by borough (lad22nm) to create borough boundaries
borough_boundaries = all_msoas.dissolve(by='lad22nm', as_index=False)
borough_boundaries = borough_boundaries.rename(columns={'lad22nm': 'Borough'})
borough_boundaries['Borough'] = borough_boundaries['Borough'].str.title()

print(f"Created {len(borough_boundaries)} borough boundaries")
print(f"Boroughs: {sorted(borough_boundaries['Borough'].tolist())}")

# Add borough boundaries to map with outlines only
for idx, row in borough_boundaries.iterrows():
    folium.GeoJson(
        data=gpd.GeoSeries([row.geometry]).__geo_interface__,
        style_function=lambda x: {
            'fillColor': 'none',
            'color': '#666666',
            'weight': 1.5,
            'fillOpacity': 0,
            'dashArray': '5, 5'
        },
        highlight_function=lambda x: {
            'fillColor': 'none',
            'color': '#000000',
            'weight': 2.5,
            'fillOpacity': 0
        },
        popup=folium.Popup(f"<b>{row['Borough']}</b>", max_width=200),
        name=f"Borough: {row['Borough']}"
    ).add_to(m)

print("Borough boundaries added")

In [5]:
# Add legend
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; right: 50px; width: 200px; height: auto; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;
     padding: 10px">
     <p style="margin:0; font-weight:bold;">London Boundaries</p>
     <p style="margin:10px 0;"><span style="background-color: #FF6B6B; padding: 8px 12px; border-radius: 3px;">&nbsp;&nbsp;&nbsp;</span> Inner London</p>
     <p style="margin:10px 0;"><span style="background-color: #4D96FF; padding: 8px 12px; border-radius: 3px;">&nbsp;&nbsp;&nbsp;</span> Outer London</p>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))
print("Legend added")

Legend added


In [6]:
# Add layer control and display
folium.LayerControl().add_to(m)

print("Map complete!")
m

Map complete!


In [7]:
# Save the map
output_path = "london_inner_outer_boundaries.html"
m.save(output_path)
print(f"Map saved to {output_path}")

Map saved to london_inner_outer_boundaries.html


In [8]:
# Display statistics
print("\n=== London Boundaries Summary ===")
print(london_boundaries[['Boundary', 'Area_Ha', 'Source']].to_string(index=False))
print(f"\nTotal area: {london_boundaries['Area_Ha'].sum():,.0f} Ha")
print(f"CRS: {london_boundaries.crs}")


=== London Boundaries Summary ===
    Boundary       Area_Ha                         Source
Inner London  34863.295694 London Plan Consultation Draft
Outer London 124606.812118 London Plan Consultation Draft

Total area: 159,470 Ha
CRS: EPSG:4326
